# Data Preparation — Deteção de Ransomware/Malware (versão corrigida)
Projeto IAAC — CRISP-ML: fases de **Data Understanding** (verificações que condicionam o treino) e **Data Preparation**, a partir de `data/raw/ransom.csv`.

### O que mudou face à versão anterior
| # | Problema na versão anterior | Correção |
|---|---|---|
| 1 | `train_test_split` estratificado pôs cópias do mesmo binário em treino e teste (fuga de informação, métricas infladas). | Split **agrupado**: grupo = componente ligada por `md5`, `sha1` ou impressão digital das features estáticas. |
| 2 | `md5`/`sha1` não são identificadores fiáveis (1 sha1 ↔ 41 md5; 1 md5 ↔ 142 sha1). | A chave de grupo não depende só dos hashes. |
| 3 | Features estáticas e dinâmicas misturadas num único `X`. Os contadores dinâmicos só existem depois de executar o ficheiro. | Dois conjuntos explícitos: **A (estático, bloqueio em tempo real)** e **B (estático + dinâmico, triagem/família)** em `feature_sets.json`. |
| 4 | `log1p`, medianas e seleção de colunas calculados com o dataset todo; `log1p` aplicado também a colunas binárias. | Tudo ajustado **só no treino**; `log1p` só em colunas contínuas. |
| 5 | Linhas duplicadas e rótulos contraditórios ignorados. | 113 duplicados exatos removidos; conflitos de rótulo sinalizados (`label_conflict_static`). |
| 6 | `OperatingSystemVersion`/`ImageVersion` com formatos inconsistentes (`0` vs `0.0`, `4` vs `4.0`), que funcionam como pista de *como* o ficheiro foi recolhido, não de maldade. Há ainda valores corrompidos (`1.37E+06`). | Normalização do formato + agrupamento de níveis raros em `other`. |
| 7 | Sem validação de que o pipeline funciona. | Secção final com baseline de sanidade (aleatório vs agrupado vs famílias novas). |

O conjunto de teste **não é usado** neste notebook para decisões. Fica guardado para a avaliação final.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import ast, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW_PATH = Path("../data/raw/ransom.csv")
OUT_DIR = Path("../data/prepared"); OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

raw = pd.read_csv(RAW_PATH)
print("Shape:", raw.shape, "| valores em falta:", int(raw.isna().sum().sum()))

Shape: (21752, 77) | valores em falta: 0


## 1. Data understanding — o que condiciona o treino
### 1.1 Alvos e tipos de colunas

In [2]:
ID_COLS = ["md5", "sha1"]
TARGETS = ["Class", "Category", "Family"]

DYN_RAW = [c for c in raw.columns if pd.api.types.is_float_dtype(raw[c])]          # contadores da sandbox
CONST_COLS = [c for c in raw.columns if raw[c].nunique() <= 1]                     # sem variância
STATIC_RAW = [c for c in raw.columns if c not in DYN_RAW + ID_COLS + TARGETS + CONST_COLS]

print("Constantes:", CONST_COLS)
print(f"Estáticas: {len(STATIC_RAW)} | Dinâmicas (sandbox): {len(DYN_RAW)} (incl. constante)")
print()
print(raw["Class"].value_counts().to_string()); print()
print(pd.crosstab(raw["Category"], raw["Class"]).to_string()); print()
print("Famílias de malware:", raw.loc[raw.Class == "Malware", "Family"].nunique())
print(raw.loc[raw.Class == "Malware", "Family"].value_counts().to_string())

Constantes: ['file_extension', 'network_threats']
Estáticas: 53 | Dinâmicas (sandbox): 18 (incl. constante)

Class
Benign     10876
Malware    10876

Class       Benign  Malware
Category                   
Benign       10876        0
RAT              0     2647
Ransomware       0     4762
Stealer          0     2018
Trojan           0     1449

Famílias de malware: 26
Family
Phobos        550
Snake         527
NanoCore      520
Raccoon       518
Remcos        512
njRat         506
Dharma        505
Shade         504
RedLine       500
WannaCry      497
Formbook      473
Gh0st         454
Ryuk          449
Guloader      444
LockBit       438
Gandcrab      382
REvil         347
Glupteba      346
Emotet        346
Agenttesla    337
DarkSide      335
Cerber        332
Qbot          322
Ursnif        309
Maze          290
Ragnar        133


### 1.2 Identificadores: os hashes repetem-se e são incoerentes
Há 21 752 linhas mas muito menos `md5` distintos, e as repetições **não** são cópias: são execuções diferentes na sandbox (os contadores dinâmicos mudam). Além disso, `md5` e `sha1` não se correspondem 1-para-1.

In [3]:
vc = raw["md5"].value_counts()
print("md5 distintos:", raw.md5.nunique(), "| sha1 distintos:", raw.sha1.nunique(), "| linhas:", len(raw))
print("Linhas em md5 repetidos:", int((raw.md5.map(vc) > 1).sum()))
print("md5 mais repetidos:", vc.head(4).to_dict())
print("máx. sha1 por md5:", raw.groupby("md5").sha1.nunique().max(), "| máx. md5 por sha1:", raw.groupby("sha1").md5.nunique().max())
print("md5 com rótulo Class contraditório:", int((raw.groupby("md5").Class.nunique() > 1).sum()))
print("Linhas totalmente duplicadas (sem hashes):", int(raw.drop(columns=ID_COLS).duplicated().sum()))

md5 distintos: 13903 | sha1 distintos: 13904 | linhas: 21752
Linhas em md5 repetidos: 10221
md5 mais repetidos: {'84c82835a5d21bbcf75a61706d8ab549': 453, '0bca3f16dd527b4150648ec1e36cb22a': 326, '8b6bc16fd137c09a08b02bbe1bb7d670': 157, 'f83fb9ce6a83da58b20685c1d7e1e546': 157}
máx. sha1 por md5: 142 | máx. md5 por sha1: 41
md5 com rótulo Class contraditório: 17
Linhas totalmente duplicadas (sem hashes): 113


### 1.3 Pistas de viés de recolha
Alguns valores de cabeçalho dizem **como** o ficheiro foi extraído e não se é malicioso. Um modelo estático pode aprendê-los e falhar fora deste dataset.

In [4]:
for c in ["PEType", "Magic", "Subsystem", "OperatingSystemVersion", "ImageVersion"]:
    t = pd.crosstab(raw[c], raw["Class"], normalize="index").round(2)
    t["n"] = raw[c].value_counts()
    print(f"--- {c}"); print(t.sort_values("n", ascending=False).head(8).to_string())
print("\nMalware com cabeçalho zerado (Magic == 0):", int(((raw.Magic.astype(str) == "0") & (raw.Class == "Malware")).sum()),
      "| benignos com Magic == 0:", int(((raw.Magic.astype(str) == "0") & (raw.Class == "Benign")).sum()))
print("\nValores corrompidos tipo notação científica em ImageVersion:",
      sorted(v for v in raw.ImageVersion.astype(str).unique() if "E+" in v))

--- PEType
Class             Benign  Malware      n
PEType                                  
PE32                0.45     0.55  19621
PE32+               0.97     0.03   2129
Unknown (0x2000)    1.00     0.00      2
--- Magic
Class  Benign  Malware      n
Magic                        
PE32     0.47     0.53  19688
PE32+    0.98     0.02   1727
0        0.00     1.00    337
--- Subsystem
Class                        Benign  Malware      n
Subsystem                                          
IMAGE_SUBSYSTEM_WINDOWS_GUI    0.42     0.58  17634
IMAGE_SUBSYSTEM_WINDOWS_CUI    0.94     0.06   3687
0                              0.00     1.00    337
IMAGE_SUBSYSTEM_NATIVE         1.00     0.00     90
IMAGE_SUBSYSTEM_UNKNOWN        1.00     0.00      4
--- OperatingSystemVersion
Class                   Benign  Malware      n
OperatingSystemVersion                        
4                         0.54     0.46  10448
5.1                       0.24     0.76   4381
6                         0.75 

### 1.4 Sinal dinâmico (e porque não serve para bloqueio em tempo real)

In [5]:
dyn_nc = [c for c in DYN_RAW if c not in CONST_COLS]
print(raw.groupby("Class")[dyn_nc].median().T.to_string()); print()
print(raw.groupby("Category")[["files_suspicious", "processes_malicious", "apis", "registry_write"]].median().to_string())

Class                 Benign  Malware
registry_read           29.0    778.5
registry_write           0.0      0.0
registry_delete          0.0      0.0
registry_total          26.0    855.0
network_dns              0.0      1.0
network_http             0.0      0.0
network_connections      0.0      3.0
processes_malicious      0.0      2.0
processes_suspicious     0.0      0.0
processes_monitored      1.0      5.0
total_procsses          35.0     41.0
files_malicious          0.0      1.0
files_suspicious         0.0      3.0
files_text               0.0      2.0
files_unknown            0.0      0.0
dlls_calls               2.0      3.0
apis                    16.0     82.0

            files_suspicious  processes_malicious   apis  registry_write
Category                                                                
Benign                   0.0                  0.0   16.0             0.0
RAT                      0.0                  2.0   44.0             0.0
Ransomware             

## 2. Limpeza
- Remove `md5`/`sha1` (identificadores) e as constantes (`file_extension`, `network_threats`). Os hashes são guardados à parte só para construir os grupos.
- Remove **duplicados exatos** (todas as colunas menos os hashes, incluindo rótulos): não trazem informação e inflacionam a avaliação.
- **Não** se removem as repetições do mesmo `md5` com contadores diferentes: são observações legítimas, mas ficam sempre no mesmo grupo (secção 3).

In [6]:
feat_cols = [c for c in raw.columns if c not in ID_COLS]
dup_mask = raw.duplicated(subset=feat_cols, keep="first")
print("Duplicados exatos removidos:", int(dup_mask.sum()))
df = raw.loc[~dup_mask].reset_index(drop=True)
print("Shape:", df.shape)

Duplicados exatos removidos: 113
Shape: (21639, 77)


## 3. Grupos anti-fuga
Duas linhas pertencem ao mesmo grupo se partilham `md5`, `sha1` **ou** as mesmas features estáticas (as mesmas características do cabeçalho PE = o mesmo binário). Os grupos são as componentes ligadas desse grafo (union-find). Todo o split e toda a validação cruzada usam estes grupos.

In [7]:
def build_groups(keys_list, n):
    parent = np.arange(n)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for keys in keys_list:
        first = {}
        for i, k in enumerate(keys):
            j = first.setdefault(k, i)
            if j != i:
                parent[find(i)] = find(j)
    return np.array([find(i) for i in range(n)])

static_fp = pd.util.hash_pandas_object(df[STATIC_RAW].astype(str), index=False).values
df["group"] = pd.factorize(build_groups([df.md5.values, df.sha1.values, static_fp], len(df)))[0]
df["group_size"] = df.groupby("group")["group"].transform("size")

# mesma impressão digital estática com Class contraditória -> ruído irredutível para o modelo estático
df["label_conflict_static"] = (df.groupby(static_fp)["Class"].transform("nunique") > 1).astype(int)

print("Grupos:", df.group.nunique(), "| maiores:", df.group_size.drop_duplicates().nlargest(4).tolist())
print("Linhas com conflito de rótulo (features estáticas idênticas, Class diferente):", int(df.label_conflict_static.sum()))

fam_tab = df[df.Class == "Malware"].groupby("Family").agg(linhas=("group", "size"), grupos=("group", "nunique"))
fam_tab["linhas_por_grupo"] = (fam_tab.linhas / fam_tab.grupos).round(1)
print("\nAmostras realmente distintas por família (grupos):")
print(fam_tab.sort_values("grupos").T.to_string())

Grupos: 11455 | maiores: [4464, 406, 166, 157]
Linhas com conflito de rótulo (features estáticas idênticas, Class diferente): 304

Amostras realmente distintas por família (grupos):
Family            Ragnar  WannaCry  Dharma  DarkSide   Maze  Cerber  LockBit  REvil   Ryuk  Gandcrab  Shade  Phobos  Glupteba  Agenttesla  Formbook   Qbot  Ursnif  Emotet  Gh0st  Guloader  NanoCore  njRat  Remcos  RedLine  Raccoon  Snake
linhas             133.0     497.0   505.0     335.0  290.0   332.0    438.0  347.0  449.0     342.0  502.0   550.0     346.0       336.0     470.0  322.0   309.0   346.0  453.0     439.0     520.0  506.0   511.0    500.0    518.0  525.0
grupos              13.0      29.0    32.0      33.0   35.0    56.0     71.0   78.0  124.0     138.0  217.0   223.0     228.0       241.0     245.0  249.0   257.0   264.0  291.0     403.0     415.0  438.0   440.0    468.0    486.0  499.0
linhas_por_grupo    10.2      17.1    15.8      10.2    8.3     5.9      6.2    4.4    3.6       2.5    

**Leitura:** o número de *grupos* é o número efetivo de amostras independentes. Famílias com poucos grupos (por exemplo WannaCry, que é quase um único binário executado centenas de vezes) não permitem tirar conclusões fiáveis ao nível da família. Reporta sempre `Family` com esta ressalva.

## 4. Formatos inconsistentes nas versões
`4.0` e `4` são a mesma versão; níveis com menos de 30 amostras (incluindo valores corrompidos como `1.37E+06`) passam a `other`. A contagem de níveis não usa rótulos.

In [8]:
def norm_version(v):
    s = str(v).strip()
    while s.endswith(".0"):
        s = s[:-2]
    return s

ONEHOT_COLS = ["PEType", "MachineType", "magic_number", "Magic", "Subsystem", "OperatingSystemVersion", "ImageVersion"]
for c in ["OperatingSystemVersion", "ImageVersion"]:
    df[c] = df[c].map(norm_version)

RARE_MIN = 30
for c in ONEHOT_COLS:
    counts = df[c].value_counts()
    rare = counts[counts < RARE_MIN].index
    df[c] = df[c].where(~df[c].isin(rare), "other")
    print(f"{c}: {df[c].nunique()} níveis")

PEType: 3 níveis
MachineType: 3 níveis
magic_number: 2 níveis
Magic: 3 níveis
Subsystem: 5 níveis
OperatingSystemVersion: 10 níveis
ImageVersion: 8 níveis


## 5. Construção das features
- **Hex → número** (`float`, porque há valores de 64 bits). Sufixos como `(Section: .text)` são ignorados.
- **Listas de flags → multi-hot**.
- **Categóricas → one-hot**.
- **Rácios dinâmicos** (os que o canvas propõe), com divisão segura.

In [9]:
def hex_to_float(val):
    if pd.isna(val):
        return np.nan
    try:
        return float(int(str(val).split(" ")[0], 16))
    except ValueError:
        return np.nan

LIST_COLS = [c for c in ["DllCharacteristics", "text_Characteristics", "rdata_Characteristics"] if c in df.columns]
HEX_COLS = [c for c in STATIC_RAW if c not in LIST_COLS + ONEHOT_COLS]
leftover = [c for c in HEX_COLS if not df[c].astype(str).str.startswith("0x").any()]
assert not leftover, f"Colunas estáticas não hexadecimais por tratar: {leftover}"

hex_df = df[HEX_COLS].apply(lambda s: s.map(hex_to_float))
print(f"{len(HEX_COLS)} colunas hexadecimais | NaN após conversão:", int(hex_df.isna().sum().sum()))

def parse_list_str(val):
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

blocks = []
for c in LIST_COLS:
    parsed = df[c].map(parse_list_str)
    flags = sorted({f for row in parsed for f in row})
    blocks.append(pd.DataFrame({f"{c}__{f}": parsed.map(lambda r, f=f: int(f in r)) for f in flags}))
    print(f"{c}: {len(flags)} flags")
multi_hot = pd.concat(blocks, axis=1)
one_hot = pd.get_dummies(df[ONEHOT_COLS], prefix=ONEHOT_COLS, dtype=int)

dyn_df = df[[c for c in DYN_RAW if c not in CONST_COLS]].copy()
def safe_ratio(a, b):
    return (a / b.where(b > 0)).fillna(0.0)
dyn_df["files_total"] = df[["files_malicious", "files_suspicious", "files_text", "files_unknown"]].sum(axis=1)
dyn_df["files_malicious_ratio"] = safe_ratio(df.files_malicious, dyn_df["files_total"])
dyn_df["processes_malicious_ratio"] = safe_ratio(df.processes_malicious, df.total_procsses)
dyn_df["registry_write_ratio"] = safe_ratio(df.registry_write, df.registry_total)

static_df = pd.concat([hex_df, multi_hot, one_hot], axis=1)
const = [c for c in static_df.columns if static_df[c].nunique() <= 1]
static_df = static_df.drop(columns=const)
print("Colunas estáticas constantes removidas:", len(const))

X = pd.concat([static_df, dyn_df], axis=1)
STATIC_FEATURES = static_df.columns.tolist()
DYNAMIC_FEATURES = dyn_df.columns.tolist()
print(f"X: {X.shape} | estáticas: {len(STATIC_FEATURES)} | dinâmicas: {len(DYNAMIC_FEATURES)}")

43 colunas hexadecimais | NaN após conversão: 0
DllCharacteristics: 13 flags
text_Characteristics: 37 flags
rdata_Characteristics: 38 flags
Colunas estáticas constantes removidas: 0
X: (21639, 186) | estáticas: 165 | dinâmicas: 21


## 6. Split agrupado
1. **Grupos muito grandes** (> 1% do dataset: repetições do mesmo ficheiro em massa) ficam **sempre no treino** e nunca em teste nem em validação. Se entrassem num fold dominariam a métrica.
2. **Teste final (~20% das linhas elegíveis)**: grupos inteiros atribuídos ao teste com quota de ~20% por família (Benign incluído); nenhum grupo é partilhado com o treino. Famílias cuja maior parte está num só grupo (ex. Maze, WannaCry) ficam acima/abaixo dos 20%.
3. **Validação cruzada no treino** (`cv_fold`, 5 folds), também agrupada e estratificada por `Family`. `cv_fold = -1` significa "sempre no treino".
4. **Validação por famílias novas** (`family_fold`): cada família de malware inteira cai num só fold, para medir generalização a variantes nunca vistas. Os benignos são repartidos por grupo.

In [10]:
y_family = df["Family"].values
groups = df["group"].values
big = (df["group_size"] > 0.01 * len(df)).values
print("Linhas em grupos grandes (sempre treino):", int(big.sum()), f"({big.mean():.1%})")

elig = np.where(~big)[0]

# Teste: atribuição gulosa de GRUPOS inteiros com quota de ~20% por família (Benign incluído).
# (o StratifiedGroupKFold não equilibrava bem o Benign: ficava só com ~13% dos benignos no teste)
TEST_FRAC = 0.20
g_codes, g_uni = pd.factorize(groups[elig])
f_codes, f_uni = pd.factorize(y_family[elig])
counts = np.zeros((len(g_uni), len(f_uni)), dtype=int)
np.add.at(counts, (g_codes, f_codes), 1)
target = counts.sum(0) * TEST_FRAC
cur = np.zeros(len(f_uni))
to_test = np.zeros(len(g_uni), dtype=bool)
for g in np.random.RandomState(SEED).permutation(len(g_uni)):
    c = counts[g]; used = c > 0
    if np.all(cur[used] + c[used] <= target[used] * 1.10):
        to_test[g] = True; cur += c
test_idx = elig[to_test[g_codes]]
train_idx = np.setdiff1d(np.arange(len(df)), test_idx)
assert not set(groups[train_idx]) & set(groups[test_idx]), "grupos partilhados entre treino e teste!"

# folds de CV no treino (grupos grandes ficam a -1)
cv_fold = np.full(len(df), -1)
train_elig = np.setdiff1d(elig, test_idx)
for k, (_, va) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(train_elig, y_family[train_elig], groups[train_elig])):
    cv_fold[train_elig[va]] = k

# folds por família inteira
rng = np.random.RandomState(SEED)
fams = [f for f in np.unique(y_family) if f != "Benign"]
rng.shuffle(fams)
fam_to_fold = {f: i % 5 for i, f in enumerate(fams)}
ben_groups = np.unique(groups[train_elig][y_family[train_elig] == "Benign"])
ben_fold = dict(zip(ben_groups, rng.randint(0, 5, len(ben_groups))))
family_fold = np.full(len(df), -1)
for i in train_elig:
    family_fold[i] = ben_fold[groups[i]] if y_family[i] == "Benign" else fam_to_fold[y_family[i]]

# fold aleatório (só para comparar na secção 9; NÃO usar em avaliação real)
rand_fold = np.full(len(df), -1)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train_elig, y_family[train_elig])):
    rand_fold[train_elig[va]] = k

print(f"Treino: {len(train_idx)} | Teste: {len(test_idx)} ({len(test_idx)/len(df):.1%})")
summary = pd.DataFrame({
    "treino": df.loc[train_idx, "Class"].value_counts(normalize=True),
    "teste": df.loc[test_idx, "Class"].value_counts(normalize=True)}).round(3)
print(summary.to_string()); print()
fam_split = pd.DataFrame({"elegíveis": pd.Series(y_family[elig]).value_counts(),
                          "no teste": df.loc[test_idx, "Family"].value_counts()}).fillna(0).astype(int)
fam_split["% no teste"] = (100 * fam_split["no teste"] / fam_split["elegíveis"]).round(1)
print("Famílias: linhas elegíveis (fora dos grupos grandes) e quota que foi para o teste:")
print(fam_split.sort_values("% no teste").T.to_string())
print("\nTamanho dos folds de CV:", pd.Series(cv_fold[cv_fold >= 0]).value_counts().sort_index().to_dict())
print("Class por fold:\n", pd.crosstab(cv_fold[cv_fold >= 0], df.loc[cv_fold >= 0, "Class"]).to_string())

Linhas em grupos grandes (sempre treino): 4870 (22.5%)
Treino: 17960 | Teste: 3679 (17.0%)
         treino  teste
Class                 
Benign    0.519  0.408
Malware   0.481  0.592

Famílias: linhas elegíveis (fora dos grupos grandes) e quota que foi para o teste:
            Agenttesla   Qbot   Maze  DarkSide  Dharma  Ragnar  Ursnif  Gh0st  NanoCore  LockBit  Guloader  Snake  Raccoon  Shade  Gandcrab  Formbook  WannaCry  njRat  Remcos  Phobos  Benign  Glupteba  Cerber  Emotet   Ryuk  RedLine  REvil
elegíveis        336.0  322.0  290.0     335.0   101.0   133.0   307.0  453.0     520.0    438.0     439.0  525.0    516.0  502.0     342.0     470.0      32.0  506.0   511.0   550.0  6825.0     346.0   332.0   346.0  446.0    500.0  346.0
no teste          73.0   70.0   63.0      73.0    22.0    29.0    67.0   99.0     114.0     96.0      96.0  115.0    113.0  110.0      75.0     103.0       7.0  111.0   112.0   121.0  1501.0      76.0    73.0    76.0   98.0    110.0   76.0
% no teste   

## 7. Imputação e `log1p` ajustados só no treino
`log1p` apenas em colunas contínuas (mais de 2 valores distintos), não negativas e com assimetria > 1 **medida no treino**. Colunas binárias (multi-hot/one-hot) não se transformam. Para modelos de árvores o `log1p` é irrelevante (transformação monótona), mas ajuda a regressão logística. Para esta, lembra-te de acrescentar `StandardScaler` no pipeline de modelação.

In [11]:
Xtr = X.iloc[train_idx]
medians = Xtr.median()
X = X.fillna(medians)
Xtr = X.iloc[train_idx]

continuous = [c for c in X.columns if Xtr[c].nunique() > 2]
skewed = [c for c in continuous if Xtr[c].min() >= 0 and Xtr[c].skew() > 1]
X[skewed] = np.log1p(X[skewed].clip(lower=0))
print(f"{len(continuous)} colunas contínuas | {len(skewed)} com log1p | NaN restantes: {int(X.isna().sum().sum())}")

64 colunas contínuas | 63 com log1p | NaN restantes: 0


## 8. Guardar datasets e metadados
- `data/prepared/train.csv` e `data/prepared/test.csv`: features + `Class`, `Category`, `Family` + `group`, `group_size`, `label_conflict_static`, `cv_fold`, `family_fold`.
- `data/prepared/feature_sets.json`: listas de colunas dos conjuntos **A** (estático) e **B** (estático + dinâmico).

Para treinar: usa `train.csv`, escolhe as colunas de `feature_sets.json`, valida com `cv_fold` (ou `family_fold`) e só no fim avalia em `test.csv`.

In [12]:
meta = pd.DataFrame({"group": df.group, "group_size": df.group_size, "label_conflict_static": df.label_conflict_static,
                     "cv_fold": cv_fold, "family_fold": family_fold, "rand_fold": rand_fold})
full = pd.concat([X, df[TARGETS], meta], axis=1)
train = full.iloc[train_idx].reset_index(drop=True)
test = full.iloc[test_idx].drop(columns=["cv_fold", "family_fold", "rand_fold"]).reset_index(drop=True)
train.to_csv(OUT_DIR / "train.csv", index=False)
test.to_csv(OUT_DIR / "test.csv", index=False)

feature_sets = {
    "A_static": STATIC_FEATURES,
    "B_static_dynamic": STATIC_FEATURES + DYNAMIC_FEATURES,
    "dynamic_only": DYNAMIC_FEATURES,
    "notes": {
        "A_static": "Disponível antes de executar o ficheiro: usar para bloqueio em tempo real.",
        "B_static_dynamic": "Inclui contadores de sandbox: só após detonação. Triagem e atribuição de família.",
        "log1p_columns": skewed,
        "imputation": "mediana do treino",
        "cv_fold": "-1 = grupo grande, sempre no treino",
    },
}
json.dump(feature_sets, open(OUT_DIR / "feature_sets.json", "w"), indent=2)
print("train:", train.shape, "| test:", test.shape)
print("A:", len(STATIC_FEATURES), "features | B:", len(STATIC_FEATURES + DYNAMIC_FEATURES), "features")

train: (17960, 195) | test: (3679, 192)
A: 165 features | B: 186 features


## 9. Sanidade: o pipeline faz o que dizemos?
Baseline rápido (gradient boosting) apenas no **treino**, para confirmar três coisas:
1. Split aleatório dá métricas mais otimistas que o agrupado → a fuga existia.
2. O conjunto A (estático) é bem mais fraco que o B, e cai muito com famílias novas.
3. As métricas de decisão do canvas: recall a FPR ≤ 1%, PR-AUC.

Os valores não são resultados finais. Servem para validar a preparação.

In [13]:
def recall_at_fpr(y, p, max_fpr=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    return float(tpr[fpr <= max_fpr].max())

y_tr = (train["Class"] == "Malware").astype(int).values

def cv_eval(cols, fold_col):
    p = np.full(len(train), np.nan)
    for k in range(5):
        va = (train[fold_col] == k).values
        m = HistGradientBoostingClassifier(max_iter=150, random_state=SEED).fit(train.loc[~va, cols], y_tr[~va])
        p[va] = m.predict_proba(train.loc[va, cols])[:, 1]
    ok = ~np.isnan(p)
    return roc_auc_score(y_tr[ok], p[ok]), average_precision_score(y_tr[ok], p[ok]), recall_at_fpr(y_tr[ok], p[ok])

rows = []
for set_name in ["A_static", "dynamic_only", "B_static_dynamic"]:
    cols = feature_sets[set_name]
    for scheme, fold_col in [("aleatório (errado)", "rand_fold"), ("agrupado", "cv_fold"), ("famílias novas", "family_fold")]:
        auc, pr, rec = cv_eval(cols, fold_col)
        rows.append({"conjunto": set_name, "validação": scheme, "ROC-AUC": round(auc, 4), "PR-AUC": round(pr, 4), "recall@FPR1%": round(rec, 4)})
print(pd.DataFrame(rows).to_string(index=False))

        conjunto          validação  ROC-AUC  PR-AUC  recall@FPR1%
        A_static aleatório (errado)   0.9801  0.9859        0.7204
        A_static           agrupado   0.9733  0.9805        0.6170
        A_static     famílias novas   0.8751  0.9117        0.2253
    dynamic_only aleatório (errado)   0.9985  0.9985        0.9952
    dynamic_only           agrupado   0.9977  0.9972        0.9933
    dynamic_only     famílias novas   0.9975  0.9977        0.9851
B_static_dynamic aleatório (errado)   0.9987  0.9988        0.9970
B_static_dynamic           agrupado   0.9986  0.9985        0.9964
B_static_dynamic     famílias novas   0.9978  0.9983        0.9937


## 10. Próximos passos (modelação)
1. Baseline: regressão logística (com `StandardScaler`) nos conjuntos A e B.
2. Random Forest / XGBoost / LightGBM, com `cv_fold` para escolher hiperparâmetros.
3. `Category` (5 classes) e depois `Family` (26 classes), com `class_weight` equilibrado.
4. Reportar sempre: recall a FPR ≤ 1%, PR-AUC, recall de Ransomware e o resultado com `family_fold`.
5. Só no fim, avaliar uma vez em `data/prepared/test.csv`.
6. Rever o impacto das colunas suspeitas de viés de recolha (secção 1.3): treinar com e sem `Magic`, `magic_number`, `Subsystem`, `OperatingSystemVersion` e `ImageVersion`.